In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import cross_validate

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
    brier_score_loss,
)
from sklearn.metrics import precision_recall_curve
from sklearn.calibration import calibration_curve

from lightgbm import LGBMClassifier
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import RandomOverSampler

import optuna
import mlflow
import mlflow.sklearn
import shap
from fairlearn.metrics import MetricFrame, demographic_parity_difference, equalized_odds_difference
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 30)

In [ ]:
# Need to display all data
pd.set_option("display.max_columns", None)
# pd.set_option("display.max_rows", None) 

# Importing the data

In [ ]:
from sklearn.datasets import fetch_openml
# Option 3 -- Diabetes 130 Hospitals
diabetes = fetch_openml(data_id=4541, as_frame=True)
X, y = diabetes.data, diabetes.target


# 1. Problem Framing

Identify the target, the number of classes and what does each class mean business Wise.

# Target Analysis

In [ ]:
target_count = y.value_counts()
target_count

We can say that for the target, it will be the *probability* of someone coming back to the hospital whether or not, whether before or after 30 days.
The original target has three categories (NO, >30, and <30); we will convert it into a binary outcome that identifies patients readmitted within 30 days or not because the goal is to organize the hospital places for comebacks on the short-term.

The business goal is to help hospital discharge teams identify patients at higher risk and prioritize them for additional follow-up when resources are limited.
That way they will be able to organize the places according to the tension they have on it.

We therefore going to define 1 as : < 30
In that case we will have prepare ourselves to a comeback of the patient.

*How is the data collected ?*

The dataset represents ten years (1999-2008) of clinical care at 130 US hospitals and integrated delivery networks. It includes over 50 features representing patient and hospital outcomes. Information was extracted from the database for encounters that satisfied the following criteria.
(1)	It is an inpatient encounter (a hospital admission).
(2)	It is a diabetic encounter, that is, one during which any kind of diabetes was entered into the system as a diagnosis.
(3)	The length of stay was at least 1 day and at most 14 days.
(4)	Laboratory tests were performed during the encounter.
(5)	Medications were administered during the encounter.




## Is our positive class rare ?

In [ ]:
df_target = pd.DataFrame(y)
rarity = len(df_target[df_target["readmitted"] == "<30"]) /len(df_target)
print(f"{round(rarity,4)*100}% of patients are readmitted within 30 days.")


P(Y = 1) = 11%

P(Y = 0) = 89 %

Prevalence = 11%

We therefore have a bit of a class imbalance even though it is not extreme.

# Look at the Features 

In [ ]:
df_diabetes= pd.DataFrame(diabetes.data)
df_diabetes 

In [ ]:
df_diabetes.info()

In [ ]:
df_diabetes["age"].value_counts()

In [ ]:
df_diabetes["weight"].value_counts()

In [ ]:
df_diabetes["insulin"].value_counts()

In [ ]:
df_diabetes.duplicated().sum()

In [ ]:
df_diabetes["patient_nbr"].duplicated().sum()

Naturally a same patient can have multiple encounters.

In [ ]:
df_diabetes["encounter_id"].duplicated().sum()

No encounter counted twice.

## 4 Stratified splits 

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = df_diabetes 
y = df_target["readmitted"]
groups = df_diabetes["patient_nbr"]

sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    sgkf.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

In [ ]:
print(y_train.value_counts(normalize=True))
print(y_test.value_counts(normalize=True))

We can see that we have almost the same proportions for the train set and the test set so we know that we can compare performances between the 2. Each of them is reprsentative of the whole dataset in terms of class distribution.

The stratified splits worked.

In [ ]:
y_train_bin = (y_train == "<30").astype(int)

## 5. Data Quality Analysis: 

Missing values, rares catgories, outliers, distributions of our values.

In [ ]:
missing = (
    X_train.isna()
    .mean()
    .sort_values(ascending=False)
)

#True = 1 , False = 0
missing[missing > 0]

In [ ]:
cat_cols = X_train.select_dtypes(include="category").columns

for col in cat_cols:
    print("\n", col)
    print(X_train[col].value_counts(dropna=False).head(10))

### Preliminary data-quality findings

- Missing information is often encoded as `"?"` or `"None"` rather than `NaN`.
- `weight` is almost entirely missing and may be removed.
- `payer_code` and `medical_specialty` contain substantial missingness but may remain informative.
We were tempted to drop `payer_code` but it can bring information on socio-economical context of the patient, his eating habits then an information on the risk of readmission. 
- `max_glu_serum` and `A1Cresult` are mostly `"None"`; we consider that it shows whether or not the patient had those tests. If not, it'a good sign for him.
- `examide` and `citoglipton` are constant and should be removed. No added value from them.
- Several medication variables are nearly constant and may be removed or consolidated.
- `age` is ordinal and can potentially be converted to interval midpoints
- `diag_1`, `diag_2`, and `diag_3` have high of unique values (options) and should probably be grouped into broader diagnostic categories rather than one-hot encoded directly.

In [ ]:
for col in cat_cols:
    freq = X_train[col].value_counts(normalize=True)
    rare = freq[freq < 0.01]  # moins de 1%
    if len(rare) > 0:
        print("\n", col)
        print(rare)

In [ ]:
#Audit on the cardinality of our features and the percentage 
# of missing values and question marks in our categorical features.
cat_cols = X_train.select_dtypes(include="category").columns

cat_summary = []

for col in cat_cols:
    counts = X_train[col].value_counts(dropna=False)
    
    cat_summary.append({
        "feature": col,
        "cardinality": X_train[col].nunique(dropna=False),
        "top_category": counts.index[0],
        "top_category_pct": counts.iloc[0] / len(X_train),
        "question_mark_pct": (X_train[col] == "?").mean()
    })

cat_summary = (
    pd.DataFrame(cat_summary)
    .sort_values("cardinality", ascending=False)
)

cat_summary

Verdict on how to handle each feature based on the audit:
- weight: drop (too many missing values)
-examide, citoglipton: drop (same values for almost all patients, no information will be extracted from it)
- Race, gender: OneHotEncoder(OHE)
- change, diabetesMed: OHE/binary
- common medication : OHE
- rare medication vars: inspect maybe drop
- payer_code: keep
medical specialty: keep
- diag_1_2_3: regroup or alternative encoding
max_glu_serum, A1Cresult: keep as categorical




# 6. Which features may contain information ?

In [ ]:
payer_analysis = (
    pd.DataFrame({
        "payer_code": X_train["payer_code"],
        "target": y_train_bin
    })
    .groupby("payer_code")
    .agg(
        n=("target", "size"),
        readmission_rate=("target", "mean")
    )
    .sort_values("readmission_rate", ascending=False)
)

payer_analysis

We can therefore see that we have different rates of redamission according to the groups of "payer_code". We will the keep it since it contains information.

In [ ]:
medication_analysis = (
    pd.DataFrame({
        "medical_specialty": X_train["medical_specialty"],
        "target": y_train_bin
    })
    .groupby("medical_specialty")
    .agg(
        n=("target", "size"),
        readmission_rate=("target", "mean")
    )
    .sort_values("readmission_rate", ascending=False)
)

medication_analysis.head(30)

we have a lot of options that represent not a lot of patients. We therefore will have to regroup some of them.

In [ ]:
# Exploratory grouping only; the modelling transformer learns this rule in each CV fold.
train_analysis = X_train.copy()
specialty_counts_eda = train_analysis["medical_specialty"].value_counts()
rare_specialties_eda = specialty_counts_eda[specialty_counts_eda < 50].index
train_analysis["medical_specialty_grouped"] = (
    train_analysis["medical_specialty"].astype(str)
    .where(~train_analysis["medical_specialty"].isin(rare_specialties_eda), "Other")
)

In [ ]:
train_analysis["target"] = y_train_bin

In [ ]:
""" A function  that will receive our dataset flagged with 0 and 1
and will return an analysis according to the categorical variable passed as an argument.
 It will return the number of observations and the readmission rate for each category of the variable."""

def categorical_target_summary(df, col):
    return (
        df.groupby(col, observed=False)["target"]
        .agg(
            n="size",
            readmission_rate="mean"
        )
        .sort_values("readmission_rate", ascending=False).head(30)
    )

In [ ]:
categorical_target_summary(
    train_analysis,
    "medical_specialty_grouped")

We have groups with small population, 

In [ ]:
num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

train_analysis.groupby("target")[num_cols].mean().T

In [ ]:

num_cols = [
    "time_in_hospital",
    "num_lab_procedures",
    "num_procedures",
    "num_medications",
    "number_outpatient",
    "number_emergency",
    "number_inpatient",
    "number_diagnoses"
]

for col in num_cols:
    train_analysis.boxplot(
        column=col,
        by="target",
        figsize=(6, 4)
    )
    
    plt.title(col)
    plt.suptitle("")
    plt.xlabel("Readmitted <30 days")
    plt.ylabel(col)
    plt.show()

The distribution are very similars between 0 and 1. We are therefore wondering if we can get info from that. We are going to keep them for now

In [ ]:
medical_summary = (
    train_analysis
    .groupby("medical_specialty_grouped", observed=True)["target"]
    .agg(
        n="size",
        readmission_rate="mean"
    )
)

medical_plot = (
    medical_summary
    .query("n >= 100")
    .sort_values("readmission_rate")
)

medical_plot["readmission_rate"].plot(
    kind="barh",
    figsize=(9, 8)
)

plt.xlabel("30-day readmission rate")
plt.ylabel("Medical specialty")
plt.title("30-day readmission rate by medical specialty (n >= 100)")
plt.show()

In [ ]:
plot_data = (
    medical_summary
    .query("n >= 100")
    .sort_values(by="readmission_rate")
)

plot_data["readmission_rate"].plot(
    kind="barh",
    figsize=(9, 8)
)

plt.axvline(0)

plt.xlabel("Difference vs global readmission rate")
plt.ylabel("Medical specialty")
plt.title("Readmission-rate difference vs overall population")
plt.show()

## 7. Data Leakage Risks

In [ ]:
len(
    set(X_train["patient_nbr"]).intersection(
        set(X_test["patient_nbr"])
    )
)

No patients in the train and the test set at the same time.
If it was the case, we could train the model and for this particular patient the model would ve seen the real answer in the train when it's supposed to be unseen data in the test set.

In [ ]:
X_train["encounter_id"].duplicated().sum()
X_test["encounter_id"].duplicated().sum()

### Which data is available at the moment of the prediction?

The prediction is assumed to be made immediately after the patient is discharged from the hospital. Therefore, all information collected during the hospital stay and known at discharge can be considered available for prediction.
This includes:
- patient demographics
- admission information
- hospital stay characteristics
- diagnoses
- laboratory results
- medication information and treatment changes
- discharge disposition
In particular, variables such as discharge_disposition_id, change, diabetesMed, A1Cresult, max_glu_serum, and medication status (Up, Down, Steady, No) are considered safe because they are known by the time the patient leaves the hospital.
The model must not use any information generated after discharge, since this would introduce data leakage.

## 8. Fairness

Fairness in ML, is meant to check if the model doesn't treat certain groups automatically into a unfavorable way just because of an sensitive attribute like gender, age or race.

In [ ]:
"""Function that will receive our dataset flagged with 0 and 1.
We redo a summary of the readmission rate by category but
with the sensitive ones s the model is not too radical according to them."""

def group_target_summary(df, group_col):
    return (
        df.groupby(group_col, observed=False)["target"]
        .agg(
            n="size",
            readmission_rate="mean"
        )
        .sort_values("readmission_rate", ascending=False)
    )

In [ ]:
group_target_summary(train_analysis, "race")

In [ ]:
group_target_summary(train_analysis, "age")

In [ ]:
group_target_summary(train_analysis, "gender")

In [ ]:
global_rate = train_analysis["target"].mean()

race_summary = group_target_summary(train_analysis, "race")
race_summary["diff_vs_global"] = (
    race_summary["readmission_rate"] - global_rate
)

gender_summary = group_target_summary(train_analysis, "gender")
gender_summary["diff_vs_global"] = (
    gender_summary["readmission_rate"] - global_rate
)

age_summary = group_target_summary(train_analysis, "age")
age_summary["diff_vs_global"] = (
    age_summary["readmission_rate"] - global_rate
)


# Visualizations of the gap between those features group and

In [ ]:

race_plot = (
    race_summary
    .sort_values("diff_vs_global")
)

plt.figure(figsize=(9, 5))

plt.plot(
    race_plot.index,
    race_plot["diff_vs_global"],
    marker="o"
)

plt.axhline(0, linestyle="--")

plt.xlabel("Race")
plt.ylabel("Difference vs global readmission rate")
plt.title("30-day readmission rate difference vs global rate by race")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:

gender_plot = (
    gender_summary
    .sort_values("diff_vs_global")
)

plt.figure(figsize=(9, 5))

plt.plot(
    gender_plot.index,
    gender_plot["diff_vs_global"],
    marker="o"
)

plt.axhline(0, linestyle="--")

plt.xlabel("Gender")
plt.ylabel("Difference vs global readmission rate")
plt.title("30-day readmission rate difference vs global rate by gender")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

We have the same reprsentativity for males and females. They both have a rate equal to the mean of the whole dataset.

In [ ]:

age_plot = (
    age_summary
    .sort_values("diff_vs_global")
)

plt.figure(figsize=(9, 5))

plt.plot(
    age_plot.index,
    age_plot["diff_vs_global"],
    marker="o"
)

plt.axhline(0, linestyle="--")

plt.xlabel("Age")
plt.ylabel("Difference vs global readmission rate")
plt.title("30-day readmission rate difference vs global rate by age")

plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

It looks like age is associated to readmission. Young patients tend to not comeback that fast when elders are more likely to return. That's normal when we know that the health is getting more unstable with age in general.

But we can see that we have an exception for the 20-30 category, the phenomenon is not monoton, "linear".


# 9. Features Engineering

In [ ]:
def group_icd9(code):
    if code in ["?", None]:
        return "Unknown"
    
    code = str(code)

    # codes commençant par V ou E
    if code.startswith("V"):
        return "Supplementary"
    if code.startswith("E"):
        return "External causes"

    try:
        num = int(float(code))
    except ValueError:
        return "Unknown"

    if 1 <= num <= 139:
        return "Infectious"
    elif 140 <= num <= 239:
        return "Neoplasms"
    elif 240 <= num <= 279:
        return "Endocrine"
    elif 280 <= num <= 289:
        return "Blood"
    elif 290 <= num <= 319:
        return "Mental"
    elif 320 <= num <= 389:
        return "Nervous"
    elif 390 <= num <= 459:
        return "Circulatory"
    elif 460 <= num <= 519:
        return "Respiratory"
    elif 520 <= num <= 579:
        return "Digestive"
    elif 580 <= num <= 629:
        return "Genitourinary"
    elif 630 <= num <= 679:
        return "Pregnancy"
    elif 680 <= num <= 709:
        return "Skin"
    elif 710 <= num <= 739:
        return "Musculoskeletal"
    elif 740 <= num <= 759:
        return "Congenital"
    elif 760 <= num <= 779:
        return "Perinatal"
    elif 780 <= num <= 799:
        return "Symptoms"
    elif 800 <= num <= 999:
        return "Injury"
    else:
        return "Other"


The ICD-9 diagnosis code ranges used in `group_icd9()` were checked against official CDC/CMS documentation to ensure that each numerical interval corresponds to the correct diagnostic family. The purpose of this function is to reduce the very high cardinality of `diag_1`, `diag_2`, and `diag_3` by mapping hundreds of individual ICD-9 codes into a smaller number of clinically meaningful disease categories. This makes the features easier to interpret and more suitable for modeling, while preserving the main medical information contained in the original diagnosis codes.

In [ ]:
for col in ["diag_1", "diag_2", "diag_3"]:
    train_analysis[col + "_group"] = train_analysis[col].apply(group_icd9)

In [ ]:
train_analysis["diag_1_group"].value_counts()

We went from 700 diseases to 19 groups. We can then use OHE.

In [ ]:
diag1_summary = (
    train_analysis
    .groupby("diag_1_group", observed=False)["target"]
    .agg(
        n="size",
        readmission_rate="mean"
    )
    .sort_values("readmission_rate", ascending=False)
)

diag1_summary

In [ ]:
diag2_summary = (
    train_analysis
    .groupby("diag_2_group", observed=False)["target"]
    .agg(
        n="size",
        readmission_rate="mean"
    )
    .sort_values("readmission_rate", ascending=False)
)

diag2_summary

Comment: Secondary diagnoses show moderate differences in 30-day readmission rates. Neoplasms, Skin, External causes, Nervous, and Genitourinary are above the overall rate, while Pregnancy is clearly lower. Since most categories have substantial sample sizes, diag_2_group appears to contain useful predictive information.

In [ ]:
diag3_summary = (
    train_analysis
    .groupby("diag_3_group", observed=False)["target"]
    .agg(
        n="size",
        readmission_rate="mean"
    )
    .sort_values("readmission_rate", ascending=False)
)

diag3_summary

Tertiary diagnoses also show meaningful variation in readmission risk. Neoplasms, Genitourinary, Skin, and Respiratory have relatively high readmission rates, whereas Pregnancy and Unknown are much lower. This suggests that diag_3_group may provide complementary information to the primary and secondary diagnosis groups.

Let's apply the diagnoses transformation with our work sample.

# 10. Modelling protocol

All experiments below use the same patient-disjoint five-fold CV on the training set.
The feature builder, imputer and encoder are fitted inside each fold. The held-out
test set stays untouched until one model and threshold have been selected.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import StratifiedGroupKFold, cross_validate, cross_val_predict
from sklearn.metrics import precision_recall_curve
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
import mlflow

EXPERIMENT_NAME = "diabetes_readmission_30d_refactored"
RANDOM_STATE = 42
N_SPLITS = 5
DROP_COLUMNS = [
    "encounter_id", "patient_nbr", "weight", "examide", "citoglipton",
    "diag_1", "diag_2", "diag_3", "medical_specialty"
]
NUMERIC_COLUMNS = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses"
]


class ReadmissionFeatures(BaseEstimator, TransformerMixin):
    """Learn the rare-specialty rule on the current fit subset only."""

    def __init__(self, min_specialty_count=50):
        self.min_specialty_count = min_specialty_count

    def fit(self, X, y=None):
        counts = X["medical_specialty"].value_counts()
        self.common_specialties_ = set(counts[counts >= self.min_specialty_count].index)
        return self

    def transform(self, X):
        result = X.copy()
        for col in ("diag_1", "diag_2", "diag_3"):
            result[col + "_group"] = result[col].map(group_icd9)
        specialty = result["medical_specialty"].astype(str)
        result["medical_specialty_grouped"] = specialty.where(
            result["medical_specialty"].isin(self.common_specialties_), "Other"
        )
        return result.drop(columns=DROP_COLUMNS)


# Only column names are needed here; every fitted model learns frequencies anew.
feature_columns = ReadmissionFeatures().fit(X_train).transform(X_train.iloc[:1]).columns
CATEGORICAL_COLUMNS = [col for col in feature_columns if col not in NUMERIC_COLUMNS]


def make_pipeline(estimator):
    preprocessing = ColumnTransformer([
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]), NUMERIC_COLUMNS),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), CATEGORICAL_COLUMNS)
    ])
    return Pipeline([
        ("features", ReadmissionFeatures()),
        ("preprocessor", preprocessing),
        ("model", estimator)
    ])


cv = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
groups_train = X_train["patient_nbr"]
assert set(groups_train).isdisjoint(set(X_test["patient_nbr"]))
for fold, (fit_idx, val_idx) in enumerate(cv.split(X_train, y_train_bin, groups_train)):
    assert set(groups_train.iloc[fit_idx]).isdisjoint(set(groups_train.iloc[val_idx]))
print("Patient overlap: 0 for the held-out test and all CV folds")
print("Features:", len(feature_columns), "numeric:", len(NUMERIC_COLUMNS),
      "categorical:", len(CATEGORICAL_COLUMNS))
mlflow.set_experiment(EXPERIMENT_NAME)

## Shared evaluation and MLflow logging

Each row below is a model evaluated with the same folds. MLflow stores the
parameters, mean and standard deviation of each score, and every fold score
as a metric step. F1, recall and precision use threshold 0.5.

In [ ]:
SCORING = {
    "roc_auc": "roc_auc",
    "pr_auc": "average_precision",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
}


def evaluate_model_cv(name, estimator, *, experiment_group="model_comparison"):
    pipeline = make_pipeline(estimator)
    scores = cross_validate(
        pipeline, X_train, y_train_bin, cv=cv, groups=groups_train,
        scoring=SCORING, n_jobs=2, error_score="raise"
    )
    result = {"model": name}
    for metric in SCORING:
        values = scores[f"test_{metric}"]
        result[metric] = float(np.mean(values))
        result[f"{metric}_std"] = float(np.std(values))

    with mlflow.start_run(run_name=name) as run:
        mlflow.set_tags({
            "evaluation": "patient_grouped_cv",
            "experiment_group": experiment_group,
            "target": "readmitted_under_30_days",
            "feature_builder": "fitted_per_fold",
        })
        mlflow.log_params({
            "model": type(estimator).__name__,
            "cv_strategy": type(cv).__name__,
            "cv_folds": N_SPLITS,
            "decision_threshold": 0.5,
            "min_specialty_count": 50,
            **{key: str(value) for key, value in estimator.get_params(deep=False).items()}
        })
        mlflow.log_metric("positive_rate", float(y_train_bin.mean()))
        for metric in SCORING:
            mlflow.log_metric(f"cv_{metric}_mean", result[metric])
            mlflow.log_metric(f"cv_{metric}_std", result[f"{metric}_std"])
            for fold, value in enumerate(scores[f"test_{metric}"]):
                mlflow.log_metric(f"fold_{metric}", float(value), step=fold)
        result["run_id"] = run.info.run_id
    return result


def run_experiments(specs, *, experiment_group):
    results = []
    for name, estimator in specs:
        print("Evaluating", name)
        results.append(evaluate_model_cv(
            name, estimator, experiment_group=experiment_group
        ))
    return pd.DataFrame(results).set_index("model").sort_values("pr_auc", ascending=False)

## Baselines and model families

The unweighted and weighted logistic models now use the same grouped CV.
All model families use the same input and preprocessing protocol.

In [ ]:
baseline_specs = [
    ("LogReg_unweighted", LogisticRegression(max_iter=1000)),
    ("LogReg_balanced", LogisticRegression(max_iter=1000, class_weight="balanced")),
    ("RF_unweighted", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=1
    )),
    ("RF_balanced", RandomForestClassifier(
        n_estimators=300, min_samples_leaf=5, class_weight="balanced",
        random_state=RANDOM_STATE, n_jobs=1
    )),
    ("LGBM_unweighted", LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        random_state=RANDOM_STATE, verbosity=-1, n_jobs=1
    )),
    ("LGBM_balanced", LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=31,
        class_weight="balanced", random_state=RANDOM_STATE, verbosity=-1, n_jobs=1
    )),
    ("XGB_weight_8", XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4, min_child_weight=5,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=float((y_train_bin == 0).sum() / (y_train_bin == 1).sum()),
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1
    )),
    ("CatBoost_balanced", CatBoostClassifier(
        iterations=500, learning_rate=0.03, depth=8, auto_class_weights="Balanced",
        loss_function="Logloss", verbose=False, random_seed=RANDOM_STATE,
        thread_count=1, allow_writing_files=False
    )),
]
baseline_results = run_experiments(baseline_specs, experiment_group="baseline")
display(baseline_results)

## Class weight sweep

This extends the comparison to weights 1, 2, 4, 6 and the training-set class
ratio. Every setting is a separate MLflow run. The previous results remain
available in the original MLflow experiment.

In [ ]:
class_ratio = float((y_train_bin == 0).sum() / (y_train_bin == 1).sum())
weight_specs = []
for weight in [1, 2, 4, 6, class_ratio]:
    weight_specs.extend([
        (f"XGB_weight_{weight:.1f}", XGBClassifier(
            n_estimators=400, learning_rate=0.05, max_depth=4, min_child_weight=5,
            subsample=0.8, colsample_bytree=0.8, scale_pos_weight=weight,
            eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=1
        )),
        (f"LGBM_weight_{weight:.1f}", LGBMClassifier(
            n_estimators=400, learning_rate=0.05, num_leaves=31,
            scale_pos_weight=weight, random_state=RANDOM_STATE, verbosity=-1, n_jobs=1
        ))
    ])
weight_results = run_experiments(weight_specs, experiment_group="weight_sweep")
display(weight_results)

## Threshold selection on out-of-fold predictions

These are **development** scores: the threshold is chosen and scored on the
same out-of-fold predictions. They are logged separately from CV scores and
must not be presented as an unbiased final result. Do not use `X_test` here.

In [ ]:
def tune_threshold_oof(name, estimator):
    probability = cross_val_predict(
        make_pipeline(estimator), X_train, y_train_bin,
        cv=cv, groups=groups_train, method="predict_proba", n_jobs=2
    )[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_train_bin, probability)
    f1 = 2 * precision[:-1] * recall[:-1] / (
        precision[:-1] + recall[:-1] + 1e-12
    )
    best = int(np.argmax(f1))
    result = {
        "model": name,
        "threshold": float(thresholds[best]),
        "oof_f1_selection": float(f1[best]),
        "oof_precision_selection": float(precision[best]),
        "oof_recall_selection": float(recall[best]),
    }
    with mlflow.start_run(run_name=f"threshold_{name}") as run:
        mlflow.set_tags({
            "evaluation": "threshold_selection_oof",
            "score_status": "selection_only_optimistic",
            "target": "readmitted_under_30_days",
        })
        mlflow.log_params({
            "model": name,
            "cv_strategy": type(cv).__name__,
            "cv_folds": N_SPLITS,
            "selected_threshold": result["threshold"],
            **{key: str(value) for key, value in estimator.get_params(deep=False).items()}
        })
        for metric in ("oof_f1_selection", "oof_precision_selection", "oof_recall_selection"):
            mlflow.log_metric(metric, result[metric])
        result["run_id"] = run.info.run_id
    return result


threshold_specs = [
    spec for spec in weight_specs
    if spec[0] in {"XGB_weight_6.0", "LGBM_weight_6.0", "LGBM_weight_1.0"}
]
threshold_results = pd.DataFrame([
    tune_threshold_oof(name, estimator) for name, estimator in threshold_specs
]).set_index("model").sort_values("oof_f1_selection", ascending=False)
display(threshold_results)

## Final held-out evaluation — run once after model and threshold selection

Choose the model using training CV and threshold-selection results, then set
`FINAL_MODEL_NAME` and run the next cell once. The code records this separate
test result in MLflow. If you change the model after seeing test scores, the
test is no longer an unbiased final estimate.

In [ ]:
FINAL_MODEL_NAME = None  # Set to an exact name from baseline_specs or weight_specs.

def evaluate_held_out(model_name, threshold):
    candidates = dict(baseline_specs + weight_specs)
    if model_name not in candidates:
        raise ValueError(f"Unknown model: {model_name}")
    estimator = make_pipeline(candidates[model_name])
    estimator.fit(X_train, y_train_bin)
    y_test_bin = (y_test == "<30").astype(int)
    probability = estimator.predict_proba(X_test)[:, 1]
    predicted = (probability >= threshold).astype(int)
    from sklearn.metrics import f1_score, precision_score, recall_score
    result = {
        "test_roc_auc": roc_auc_score(y_test_bin, probability),
        "test_pr_auc": average_precision_score(y_test_bin, probability),
        "test_f1": f1_score(y_test_bin, predicted),
        "test_precision": precision_score(y_test_bin, predicted, zero_division=0),
        "test_recall": recall_score(y_test_bin, predicted, zero_division=0),
    }
    with mlflow.start_run(run_name=f"FINAL_TEST_{model_name}"):
        mlflow.set_tags({"evaluation": "held_out_patient_test", "model_selection_complete": "true"})
        mlflow.log_params({"model": model_name, "decision_threshold": threshold})
        mlflow.log_metrics(result)
    return result

if FINAL_MODEL_NAME is not None:
    selected_threshold = (
        float(threshold_results.loc[FINAL_MODEL_NAME, "threshold"])
        if FINAL_MODEL_NAME in threshold_results.index else 0.5
    )
    display(evaluate_held_out(FINAL_MODEL_NAME, selected_threshold))